In [1]:
%%writefile harness.py
"""
Execution-accuracy harness for Text-to-SQL evaluation.

Core idea: a predicted SQL query is "correct" if executing it against the
target database returns the same result set as executing the gold SQL,
NOT if the query text matches. This is what lets us compare A/B/C/D fairly
even when there are many syntactically different but semantically correct
ways to write the same query.

Usage:
    from harness import score_dataset, execute_sql

    results = score_dataset(examples, db_root="/path/to/databases")
    print(results["execution_accuracy"], results["valid_sql_rate"])

`examples` is a list of dicts: {"db_id": ..., "gold_sql": ..., "pred_sql": ...}
"""

import sqlite3
import multiprocessing as mp
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


DEFAULT_TIMEOUT_SECONDS = 30


@dataclass
class ExecResult:
    ok: bool
    rows: Optional[list] = None
    error: Optional[str] = None


def _execute_worker(db_path: str, sql: str, conn_queue: mp.Queue):
    """Runs in a separate process so a hanging/malicious query can be killed."""
    try:
        # Read-only: open with a URI and mode=ro so nothing can be mutated.
        uri = f"file:{db_path}?mode=ro"
        conn = sqlite3.connect(uri, uri=True, timeout=5)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        conn_queue.put(("ok", rows))
    except Exception as e:
        conn_queue.put(("error", str(e)))


def execute_sql(db_path: str, sql: str, timeout: int = DEFAULT_TIMEOUT_SECONDS) -> ExecResult:
    """Execute `sql` against the SQLite file at `db_path` in an isolated,
    time-limited subprocess. Returns ExecResult(ok, rows, error)."""
    ctx = mp.get_context("spawn")
    q = ctx.Queue()
    p = ctx.Process(target=_execute_worker, args=(db_path, sql, q))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return ExecResult(ok=False, error=f"timeout after {timeout}s")

    if q.empty():
        return ExecResult(ok=False, error="process died without result (crash/segfault)")

    status, payload = q.get()
    if status == "ok":
        return ExecResult(ok=True, rows=payload)
    return ExecResult(ok=False, error=payload)


def _normalize_rows(rows: list) -> frozenset:
    """Order-insensitive, duplicate-preserving-ish comparison.

    We compare as a multiset (Counter over stringified rows) rather than a
    plain set, so a query that duplicates a row isn't incorrectly treated as
    equal to one that doesn't. Row order is ignored (most gold SQL doesn't
    enforce a specific order without ORDER BY, and even with ORDER BY, BIRD's
    own eval treats output order permissively unless the question implies it).
    """
    from collections import Counter
    stringified = [tuple(str(v) for v in row) for row in rows]
    return Counter(stringified)


def results_match(gold_rows: list, pred_rows: list) -> bool:
    return _normalize_rows(gold_rows) == _normalize_rows(pred_rows)


def score_example(example: dict, db_root: str, timeout: int = DEFAULT_TIMEOUT_SECONDS) -> dict:
    """Score a single example. Expects keys: db_id, gold_sql, pred_sql.
    Assumes SQLite files live at {db_root}/{db_id}/{db_id}.sqlite
    (BIRD/Spider convention) — adjust `_db_path` if your layout differs."""
    db_path = _db_path(db_root, example["db_id"])

    gold = execute_sql(db_path, example["gold_sql"], timeout)
    pred = execute_sql(db_path, example["pred_sql"], timeout)

    out = {
        "db_id": example["db_id"],
        "gold_sql": example["gold_sql"],
        "pred_sql": example["pred_sql"],
        "gold_valid": gold.ok,
        "pred_valid": pred.ok,
        "gold_error": gold.error,
        "pred_error": pred.error,
        "correct": False,
    }
    # Pass through any extra keys (question_id, difficulty, etc.) so callers
    # can slice results afterward without re-scoring.
    for k, v in example.items():
        if k not in ("db_id", "gold_sql", "pred_sql"):
            out[k] = v

    if not gold.ok:
        # Gold itself failed to execute -- this is a data problem, flag it
        # loudly rather than silently scoring it as incorrect.
        out["skip_reason"] = "gold_sql_failed"
        return out

    if not pred.ok:
        out["correct"] = False
        return out

    out["correct"] = results_match(gold.rows, pred.rows)
    return out


def _db_path(db_root: str, db_id: str) -> str:
    p = Path(db_root) / db_id / f"{db_id}.sqlite"
    if not p.exists():
        raise FileNotFoundError(
            f"No sqlite file at {p}. Check db_root/layout matches your dataset's actual structure."
        )
    return str(p)


def score_dataset(examples: list, db_root: str, timeout: int = DEFAULT_TIMEOUT_SECONDS,
                   n_workers: int = 1) -> dict:
    """Score a whole list of examples. Set n_workers > 1 to parallelize across
    a process pool once you trust the harness (start with 1 for debugging)."""
    per_example = []

    if n_workers <= 1:
        for ex in examples:
            per_example.append(score_example(ex, db_root, timeout))
    else:
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor(max_workers=n_workers) as pool:
            futures = [pool.submit(score_example, ex, db_root, timeout) for ex in examples]
            per_example = [f.result() for f in futures]

    scored = [r for r in per_example if r.get("skip_reason") != "gold_sql_failed"]
    skipped = [r for r in per_example if r.get("skip_reason") == "gold_sql_failed"]

    n = len(scored)
    ex_acc = sum(r["correct"] for r in scored) / n if n else 0.0
    valid_rate = sum(r["pred_valid"] for r in scored) / n if n else 0.0

    return {
        "execution_accuracy": ex_acc,
        "valid_sql_rate": valid_rate,
        "n_scored": n,
        "n_skipped_bad_gold": len(skipped),
        "per_example": per_example,
    }

Writing harness.py


In [2]:
%%writefile test_harness.py
import os
import sqlite3
import tempfile
from harness import score_dataset


def build_toy_db(path: str):
    conn = sqlite3.connect(path)
    cur = conn.cursor()
    cur.executescript("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            name TEXT,
            country TEXT
        );
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            revenue REAL,
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );
        INSERT INTO customers VALUES (1, 'Alice', 'USA');
        INSERT INTO customers VALUES (2, 'Bob', 'UK');
        INSERT INTO customers VALUES (3, 'Carla', 'USA');
        INSERT INTO orders VALUES (1, 1, 100.0);
        INSERT INTO orders VALUES (2, 1, 50.0);
        INSERT INTO orders VALUES (3, 2, 200.0);
        INSERT INTO orders VALUES (4, 3, 300.0);
    """)
    conn.commit()
    conn.close()


def main():
    tmpdir = tempfile.mkdtemp()
    db_root = os.path.join(tmpdir, "databases")
    db_id = "toy_store"
    db_dir = os.path.join(db_root, db_id)
    os.makedirs(db_dir, exist_ok=True)
    db_path = os.path.join(db_dir, f"{db_id}.sqlite")
    build_toy_db(db_path)

    gold_sql = (
        "SELECT c.name, SUM(o.revenue) as total_rev "
        "FROM customers c JOIN orders o ON c.customer_id = o.customer_id "
        "GROUP BY c.name ORDER BY total_rev DESC"
    )

    examples_gold_as_pred = [{"db_id": db_id, "gold_sql": gold_sql, "pred_sql": gold_sql}]

    examples_wrong = [
        {"db_id": db_id, "gold_sql": gold_sql, "pred_sql": "SELECT * FROM nonexistent_table"},
        {"db_id": db_id, "gold_sql": gold_sql, "pred_sql": "SELECT name FROM customers"},
    ]

    equivalent_sql = (
        "select c.name, sum(o.revenue) as total_rev from orders o "
        "join customers c on o.customer_id = c.customer_id "
        "group by c.name order by total_rev desc"
    )
    examples_equivalent = [{"db_id": db_id, "gold_sql": gold_sql, "pred_sql": equivalent_sql}]

    examples_bad_gold = [{"db_id": db_id, "gold_sql": "SELECT * FROM table_that_does_not_exist", "pred_sql": gold_sql}]

    r1 = score_dataset(examples_gold_as_pred, db_root)
    r2 = score_dataset(examples_wrong, db_root)
    r3 = score_dataset(examples_equivalent, db_root)
    r4 = score_dataset(examples_bad_gold, db_root)

    print("=== Case 1: gold-as-prediction ===")
    print(f"  execution_accuracy = {r1['execution_accuracy']:.3f}  (expect 1.000)")
    assert r1["execution_accuracy"] == 1.0

    print("=== Case 2: wrong/broken predictions ===")
    print(f"  execution_accuracy = {r2['execution_accuracy']:.3f}  (expect 0.000)")
    assert r2["execution_accuracy"] == 0.0

    print("=== Case 3: semantically-equivalent query ===")
    print(f"  execution_accuracy = {r3['execution_accuracy']:.3f}  (expect 1.000)")
    assert r3["execution_accuracy"] == 1.0

    print("=== Case 4: invalid gold SQL ===")
    print(f"  n_scored = {r4['n_scored']}  n_skipped_bad_gold = {r4['n_skipped_bad_gold']}  (expect 0 and 1)")
    assert r4["n_skipped_bad_gold"] == 1

    print("\nALL CHECKS PASSED -- harness meets the Week 1 exit criterion.")


if __name__ == "__main__":
    main()

Writing test_harness.py


In [3]:
!python test_harness.py

=== Case 1: gold-as-prediction ===
  execution_accuracy = 1.000  (expect 1.000)
=== Case 2: wrong/broken predictions ===
  execution_accuracy = 0.000  (expect 0.000)
=== Case 3: semantically-equivalent query ===
  execution_accuracy = 1.000  (expect 1.000)
=== Case 4: invalid gold SQL ===
  n_scored = 0  n_skipped_bad_gold = 1  (expect 0 and 1)

ALL CHECKS PASSED -- harness meets the Week 1 exit criterion.


In [4]:
!wget -c https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip -O /kaggle/working/dev.zip

--2026-10-06 08:06:43--  https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip
Resolving bird-bench.oss-cn-beijing.aliyuncs.com (bird-bench.oss-cn-beijing.aliyuncs.com)... 8.141.181.247
Connecting to bird-bench.oss-cn-beijing.aliyuncs.com (bird-bench.oss-cn-beijing.aliyuncs.com)|8.141.181.247|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 346207293 (330M) [application/zip]
Saving to: ‘/kaggle/working/dev.zip’

/kaggle/working/dev 100%[===================>] 330.17M  55.5MB/s    in 7.5s    

2026-10-06 08:06:53 (43.9 MB/s) - ‘/kaggle/working/dev.zip’ saved [346207293/346207293]



In [5]:
!unzip -q /kaggle/working/dev.zip -d /kaggle/working/bird_dev
!find /kaggle/working/bird_dev -maxdepth 3

/kaggle/working/bird_dev
/kaggle/working/bird_dev/dev_20240627
/kaggle/working/bird_dev/dev_20240627/dev_databases.zip
/kaggle/working/bird_dev/dev_20240627/dev.json
/kaggle/working/bird_dev/dev_20240627/dev_tied_append.json
/kaggle/working/bird_dev/dev_20240627/dev_tables.json
/kaggle/working/bird_dev/dev_20240627/dev.sql


In [6]:
import glob

# find the nested databases zip wherever it landed
nested_zips = glob.glob("/kaggle/working/bird_dev/**/dev_databases.zip", recursive=True)
print("Found:", nested_zips)

if nested_zips:
    !unzip -q "{nested_zips[0]}" -d /kaggle/working/bird_dev_databases
else:
    print("No nested zip found — dev.zip structure may have changed, paste the Cell 2 output back to me")

Found: ['/kaggle/working/bird_dev/dev_20240627/dev_databases.zip']


In [7]:
import json
import glob

dev_json_path = glob.glob("/kaggle/working/bird_dev/**/dev.json", recursive=True)[0]
with open(dev_json_path) as f:
    dev_data = json.load(f)

print(f"Total dev questions: {len(dev_data)}")
print("\nSample question:")
print(json.dumps(dev_data[0], indent=2))

sqlite_files = glob.glob("/kaggle/working/bird_dev_databases/**/*.sqlite", recursive=True)
print(f"\nTotal .sqlite database files found: {len(sqlite_files)}")
print("Example:", sqlite_files[0] if sqlite_files else "NONE FOUND")

Total dev questions: 1534

Sample question:
{
  "question_id": 0,
  "db_id": "california_schools",
  "question": "What is the highest eligible free rate for K-12 students in the schools in Alameda County?",
  "evidence": "Eligible free rate for K-12 = `Free Meal Count (K-12)` / `Enrollment (K-12)`",
  "SQL": "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1",
  "difficulty": "simple"
}

Total .sqlite database files found: 11
Example: /kaggle/working/bird_dev_databases/dev_databases/european_football_2/european_football_2.sqlite


In [8]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 8.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 6.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 12.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 22.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-stora

In [9]:
%%writefile schema_utils.py
import sqlite3

def get_schema_ddl(db_path: str) -> str:
    """Pull CREATE TABLE statements directly from the sqlite file.
    This is the 'full schema' context for System A/B — no retrieval, everything included."""
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL")
    statements = [row[0] for row in cur.fetchall()]
    conn.close()
    return ";\n\n".join(statements) + ";"

Writing schema_utils.py


In [10]:
import json, glob, random

dev_json_path = glob.glob("/kaggle/working/bird_dev/**/dev.json", recursive=True)[0]
with open(dev_json_path) as f:
    dev_data = json.load(f)

DB_ROOT = "/kaggle/working/bird_dev_databases/dev_databases"

random.seed(42)  # reproducibility -- same 200 questions every time you rerun this

by_difficulty = {}
for ex in dev_data:
    by_difficulty.setdefault(ex["difficulty"], []).append(ex)

for k, v in by_difficulty.items():
    print(f"{k}: {len(v)} available")

# proportional sample -> 200 total, keeping the difficulty mix representative
TOTAL = 200
sample = []
for diff, items in by_difficulty.items():
    n = round(TOTAL * len(items) / len(dev_data))
    sample.extend(random.sample(items, min(n, len(items))))

print(f"\nSampled {len(sample)} questions")
with open("/kaggle/working/eval_sample_200.json", "w") as f:
    json.dump(sample, f, indent=2)

simple: 925 available
moderate: 464 available
challenging: 145 available

Sampled 200 questions


In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from schema_utils import get_schema_ddl

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"  # verify this is still current before committing further

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

PROMPT_TEMPLATE = """You are a SQLite expert. Given the database schema and a question, write a single SQL query that answers it.

### Schema:
{schema}

### Question:
{question}

### Evidence:
{evidence}

Return ONLY the SQL query, no explanation, no markdown formatting.

### SQL:
"""

def generate_sql(example, max_new_tokens=256):
    schema = get_schema_ddl(f"{DB_ROOT}/{example['db_id']}/{example['db_id']}.sqlite")
    prompt = PROMPT_TEMPLATE.format(
        schema=schema, question=example["question"], evidence=example.get("evidence", "")
    )
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip().strip("```sql").strip("```").strip()

# smoke test on just 5
for ex in sample[:5]:
    pred = generate_sql(ex)
    print(f"Q: {ex['question'][:80]}...")
    print(f"Gold: {ex['SQL'][:100]}")
    print(f"Pred: {pred[:100]}")
    print("---")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

AttributeError: 

In [12]:
def generate_sql(example, max_new_tokens=256):
    schema = get_schema_ddl(f"{DB_ROOT}/{example['db_id']}/{example['db_id']}.sqlite")
    prompt = PROMPT_TEMPLATE.format(
        schema=schema, question=example["question"], evidence=example.get("evidence", "")
    )
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip().strip("```sql").strip("```").strip()

In [13]:
for ex in sample[:5]:
    pred = generate_sql(ex)
    print(f"Q: {ex['question'][:80]}...")
    print(f"Gold: {ex['SQL'][:100]}")
    print(f"Pred: {pred[:100]}")
    print("---")

Q: In terms of number of points acquired, how many victories did the driver who ran...
Gold: SELECT SUM(CASE WHEN points = 91 THEN wins ELSE 0 END) FROM driverStandings
Pred: SELECT COUNT(T2.wins) FROM results AS T1 INNER JOIN constructorStandings AS T2 ON T1.constructorId =
---
Q: How many clients who were born in 1920 stay in east Bohemia?...
Gold: SELECT COUNT(T1.client_id) FROM client AS T1 INNER JOIN district AS T2 ON T1.district_id = T2.distri
Pred: SELECT COUNT(T1.client_id) FROM client AS T1 INNER JOIN district AS T2 ON T1.district_id = T2.distri
---
Q: What is the type of education offered in the school who scored the highest avera...
Gold: SELECT T2.EdOpsName FROM satscores AS T1 INNER JOIN schools AS T2 ON T1.cds = T2.CDSCode ORDER BY T1
Pred: SELECT DISTINCT T2.EdOpsName FROM satscores AS T1 INNER JOIN schools AS T2 ON T1.cds = T2.CDSCode WH
---
Q: Provide list of patients and their diagnosis with triglyceride (TG) index greate...
Gold: SELECT T1.ID, T1.Diagnosis FROM Patie

In [14]:
from tqdm.notebook import tqdm
import json

predictions = []
errors = []

for ex in tqdm(sample, desc="Generating SQL"):
    try:
        pred_sql = generate_sql(ex)
        predictions.append({
            "db_id": ex["db_id"],
            "question_id": ex["question_id"],
            "gold_sql": ex["SQL"],
            "pred_sql": pred_sql,
            "difficulty": ex["difficulty"],
        })
    except Exception as e:
        errors.append({"question_id": ex["question_id"], "error": str(e)})

print(f"Generated: {len(predictions)}  Errors: {len(errors)}")
if errors:
    print("First few errors:", errors[:3])

with open("/kaggle/working/system_a_predictions.json", "w") as f:
    json.dump(predictions, f, indent=2)

Generating SQL:   0%|          | 0/200 [00:00<?, ?it/s]

Generated: 200  Errors: 0


In [15]:
from harness import score_dataset
from collections import defaultdict
import json

with open("/kaggle/working/system_a_predictions.json") as f:
    predictions = json.load(f)

results = score_dataset(predictions, db_root=DB_ROOT)

print(f"Execution Accuracy: {results['execution_accuracy']:.3f}")
print(f"Valid SQL Rate:     {results['valid_sql_rate']:.3f}")

# breakdown by difficulty -- now works since question_id/difficulty pass through
by_diff = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results["per_example"]:
    d = r["difficulty"]
    by_diff[d]["total"] += 1
    by_diff[d]["correct"] += r["correct"]

print("\n--- By difficulty ---")
for d, stats in by_diff.items():
    print(f"{d}: {stats['correct']}/{stats['total']} = {stats['correct']/stats['total']:.3f}")

# diagnose the invalid predictions -- truncation or genuine errors?
invalid = [r for r in results["per_example"] if not r["pred_valid"]]
print(f"\n--- {len(invalid)} invalid predictions, first 5 ---")
for r in invalid[:5]:
    print(f"Error: {r['pred_error']}")
    print(f"Pred (last 80 chars): ...{r['pred_sql'][-80:]}")
    print("---")

Execution Accuracy: 0.396
Valid SQL Rate:     0.741

--- By difficulty ---
simple: 52/121 = 0.430
moderate: 22/60 = 0.367
challenging: 4/19 = 0.211

--- 53 invalid predictions, first 5 ---
Error: no such column: T2.artist
Pred (last 80 chars): ...NER JOIN foreign_data AS T2 ON T1.uuid = T2.uuid WHERE T2.artist = 'Daren Bader'
---
Error: no such column: T1.format
Pred (last 80 chars): ...1 INNER JOIN foreign_data AS T2 ON T1.uuid = T2.uuid WHERE T1.format = 'legacy';
---
Error: no such column: UN
Pred (last 80 chars): ...SELECT ID, SEX, Birthday FROM Patient WHERE UN = 29;
---
Error: no such column: T
Pred (last 80 chars): ...ERE T-CHO = ( SELECT MAX(T-CHO) FROM Laboratory ) ORDER BY Birthday ASC LIMIT 1;
---
Error: no such column: T1.cds
Pred (last 80 chars): ... ON T1.CDSCode = T2.cds
WHERE T1.MailCity = 'Lakeport' AND T2.NumGE1500 >= 1500;
---


In [16]:
import json
with open("/kaggle/working/system_a_results_summary.json", "w") as f:
    json.dump({
        "execution_accuracy": results["execution_accuracy"],
        "valid_sql_rate": results["valid_sql_rate"],
        "n_scored": results["n_scored"],
        "by_difficulty": {d: s["correct"]/s["total"] for d, s in by_diff.items()},
    }, f, indent=2)

In [17]:
!pip install -q sqlglot rank_bm25 sentence-transformers

import glob, os

# BIRD ships per-database CSVs describing each column, IF the dataset includes them
desc_dirs = glob.glob("/kaggle/working/bird_dev_databases/dev_databases/*/database_description")
print(f"Databases with description folders: {len(desc_dirs)} / 11")

if desc_dirs:
    sample_dir = desc_dirs[0]
    print(f"\nExample: {sample_dir}")
    files = os.listdir(sample_dir)
    print(f"CSV files inside: {files[:5]}")

    import pandas as pd
    sample_csv = os.path.join(sample_dir, files[0])
    df = pd.read_csv(sample_csv)
    print(f"\nColumns in {files[0]}:")
    print(df.columns.tolist())
    print("\nFirst few rows:")
    print(df.head(3).to_string())

Databases with description folders: 11 / 11

Example: /kaggle/working/bird_dev_databases/dev_databases/european_football_2/database_description
CSV files inside: ['Country.csv', 'Team.csv', 'Player_Attributes.csv', 'Player.csv', 'League.csv']

Columns in Country.csv:
['original_column_name', 'column_name', 'column_description', 'data_format', 'value_description']

First few rows:
  original_column_name  column_name           column_description data_format  value_description
0                   id          NaN  the unique id for countries     integer                NaN
1                 name          NaN                 country name        text                NaN


In [18]:
%%writefile schema_kb.py
"""
Builds a structured knowledge base for one database: every column with its
description, every table, and the PK/FK relationships between them. This is
the raw material the retriever will search over -- get this right first,
everything downstream depends on it.
"""

import sqlite3
import pandas as pd
import os


def get_foreign_keys(db_path: str) -> list[dict]:
    """Returns [{from_table, from_col, to_table, to_col}, ...] using SQLite's
    own PRAGMA, not regex-parsing DDL -- more reliable."""
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    fks = []
    for t in tables:
        cur.execute(f"PRAGMA foreign_key_list(`{t}`)")
        for row in cur.fetchall():
            # PRAGMA foreign_key_list columns: id, seq, table, from, to, ...
            fks.append({
                "from_table": t,
                "from_col": row[3],
                "to_table": row[2],
                "to_col": row[4],
            })
    conn.close()
    return fks


def get_primary_keys(db_path: str) -> dict:
    """Returns {table_name: [pk_col1, pk_col2, ...]}"""
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    pks = {}
    for t in tables:
        cur.execute(f"PRAGMA table_info(`{t}`)")
        pk_cols = [row[1] for row in cur.fetchall() if row[5] > 0]  # row[5] = pk flag
        pks[t] = pk_cols
    conn.close()
    return pks


def get_sample_values(db_path: str, table: str, column: str, n: int = 3) -> list:
    """A few distinct real values from this column -- helps the retriever
    match questions that mention actual data (e.g. 'California' -> state column)."""
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cur = conn.cursor()
        cur.execute(f"SELECT DISTINCT `{column}` FROM `{table}` WHERE `{column}` IS NOT NULL LIMIT {n}")
        vals = [row[0] for row in cur.fetchall()]
        conn.close()
        return vals
    except Exception:
        return []


def load_descriptions(desc_dir: str) -> dict:
    """Returns {table_name: DataFrame} from the database_description CSVs.
    Handles encoding issues -- BIRD's CSVs are inconsistently encoded across databases."""
    tables = {}
    for fname in os.listdir(desc_dir):
        if not fname.endswith(".csv"):
            continue
        table_name = fname[:-4]
        path = os.path.join(desc_dir, fname)
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="latin-1")
        tables[table_name] = df
    return tables


def build_schema_kb(db_root: str, db_id: str) -> dict:
    """The main entry point. Returns a structured dict:
    {
      "tables": [table_name, ...],
      "columns": [
        {table, column, description, value_description, data_format,
         is_primary_key, sample_values}, ...
      ],
      "foreign_keys": [{from_table, from_col, to_table, to_col}, ...]
    }
    """
    db_path = f"{db_root}/{db_id}/{db_id}.sqlite"
    desc_dir = f"{db_root}/{db_id}/database_description"

    fks = get_foreign_keys(db_path)
    pks = get_primary_keys(db_path)
    descriptions = load_descriptions(desc_dir) if os.path.exists(desc_dir) else {}

    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    columns = []
    for t in tables:
        cur.execute(f"PRAGMA table_info(`{t}`)")
        table_cols = cur.fetchall()  # (cid, name, type, notnull, dflt_value, pk)

        desc_df = descriptions.get(t)

        for col_row in table_cols:
            col_name = col_row[1]
            col_type = col_row[2]

            description = ""
            value_description = ""
            if desc_df is not None:
                # match on original_column_name, strip whitespace defensively
                match = desc_df[desc_df["original_column_name"].astype(str).str.strip() == col_name]
                if len(match) > 0:
                    row = match.iloc[0]
                    description = str(row.get("column_description", "") or "")
                    value_description = str(row.get("value_description", "") or "")
                    if description == "nan":
                        description = ""
                    if value_description == "nan":
                        value_description = ""

            columns.append({
                "table": t,
                "column": col_name,
                "type": col_type,
                "description": description,
                "value_description": value_description,
                "is_primary_key": col_name in pks.get(t, []),
                "sample_values": get_sample_values(db_path, t, col_name),
            })

    conn.close()
    return {"tables": tables, "columns": columns, "foreign_keys": fks}

Writing schema_kb.py


In [19]:
from schema_kb import build_schema_kb
import json

kb = build_schema_kb(DB_ROOT, "student_club")

print(f"Tables: {kb['tables']}")
print(f"\nTotal columns indexed: {len(kb['columns'])}")
print(f"\nForeign keys found: {len(kb['foreign_keys'])}")
for fk in kb["foreign_keys"][:5]:
    print(f"  {fk['from_table']}.{fk['from_col']} -> {fk['to_table']}.{fk['to_col']}")

print("\n--- Sample column entries ---")
for col in kb["columns"][:3]:
    print(json.dumps(col, indent=2, default=str))

Tables: ['event', 'major', 'zip_code', 'attendance', 'budget', 'expense', 'income', 'member']

Total columns indexed: 48

Foreign keys found: 8
  attendance.link_to_member -> member.member_id
  attendance.link_to_event -> event.event_id
  budget.link_to_event -> event.event_id
  expense.link_to_member -> member.member_id
  expense.link_to_budget -> budget.budget_id

--- Sample column entries ---
{
  "table": "event",
  "column": "event_id",
  "type": "TEXT",
  "description": "A unique identifier for the event",
  "value_description": "",
  "is_primary_key": true,
  "sample_values": [
    "rec0Si5cQ4rJRVzd6",
    "rec0akZnLLpGUloLH",
    "rec0dZPcWXF0QjNnE"
  ]
}
{
  "table": "event",
  "column": "event_name",
  "type": "TEXT",
  "description": "event name",
  "value_description": "",
  "is_primary_key": false,
  "sample_values": [
    "March Meeting",
    "Officers meeting - January",
    "Spring Elections"
  ]
}
{
  "table": "event",
  "column": "event_date",
  "type": "TEXT",
  "desc

In [20]:
import glob

all_db_ids = sorted([os.path.basename(p) for p in glob.glob(f"{DB_ROOT}/*") if os.path.isdir(p)])
print(f"Databases found: {all_db_ids}")

all_kbs = {}
for db_id in all_db_ids:
    try:
        all_kbs[db_id] = build_schema_kb(DB_ROOT, db_id)
    except Exception as e:
        print(f"FAILED on {db_id}: {e}")

print(f"\nSuccessfully built KBs for {len(all_kbs)} / {len(all_db_ids)} databases")
for db_id, kb in all_kbs.items():
    print(f"  {db_id}: {len(kb['tables'])} tables, {len(kb['columns'])} columns, {len(kb['foreign_keys'])} FKs")

Databases found: ['california_schools', 'card_games', 'codebase_community', 'debit_card_specializing', 'european_football_2', 'financial', 'formula_1', 'student_club', 'superhero', 'thrombosis_prediction', 'toxicology']

Successfully built KBs for 11 / 11 databases
  california_schools: 3 tables, 89 columns, 2 FKs
  card_games: 7 tables, 117 columns, 4 FKs
  codebase_community: 8 tables, 71 columns, 13 FKs
  debit_card_specializing: 6 tables, 23 columns, 2 FKs
  european_football_2: 8 tables, 201 columns, 31 FKs
  financial: 8 tables, 55 columns, 8 FKs
  formula_1: 14 tables, 96 columns, 19 FKs
  student_club: 8 tables, 48 columns, 8 FKs
  superhero: 10 tables, 31 columns, 11 FKs
  thrombosis_prediction: 3 tables, 64 columns, 2 FKs
  toxicology: 4 tables, 11 columns, 5 FKs


In [21]:
%%writefile doc_builder.py
"""
Converts a schema KB into flat text documents the retriever can index.
One document per column, one per table -- this is what BM25/embeddings
actually search over.
"""

def column_to_doc(col: dict) -> str:
    """Text representation of one column, folding in everything that
    might help a retriever match it to a natural-language question."""
    parts = [f"Table: {col['table']}", f"Column: {col['column']}"]
    if col["description"]:
        parts.append(f"Description: {col['description']}")
    if col["value_description"]:
        parts.append(f"Value meaning: {col['value_description']}")
    if col["is_primary_key"]:
        parts.append("This is a primary key.")
    if col["sample_values"]:
        vals = ", ".join(str(v) for v in col["sample_values"][:3])
        parts.append(f"Example values: {vals}")
    return " | ".join(parts)


def table_to_doc(table_name: str, kb: dict) -> str:
    """Text representation of a whole table: its columns and how it
    connects to other tables via foreign keys."""
    cols_in_table = [c["column"] for c in kb["columns"] if c["table"] == table_name]
    parts = [f"Table: {table_name}", f"Columns: {', '.join(cols_in_table)}"]

    related = [fk for fk in kb["foreign_keys"]
               if fk["from_table"] == table_name or fk["to_table"] == table_name]
    if related:
        rel_strs = []
        for fk in related:
            rel_strs.append(f"{fk['from_table']}.{fk['from_col']} -> {fk['to_table']}.{fk['to_col']}")
        parts.append(f"Relationships: {'; '.join(rel_strs)}")
    return " | ".join(parts)


def build_documents(db_id: str, kb: dict) -> dict:
    """Returns {'column_docs': [{'doc_id', 'text', 'table', 'column'}, ...],
                'table_docs': [{'doc_id', 'text', 'table'}, ...]}
    doc_id is used later to map a retrieval hit back to the actual schema element."""
    column_docs = []
    for col in kb["columns"]:
        column_docs.append({
            "doc_id": f"{db_id}::{col['table']}::{col['column']}",
            "text": column_to_doc(col),
            "db_id": db_id,
            "table": col["table"],
            "column": col["column"],
        })

    table_docs = []
    for t in kb["tables"]:
        table_docs.append({
            "doc_id": f"{db_id}::{t}",
            "text": table_to_doc(t, kb),
            "db_id": db_id,
            "table": t,
        })

    return {"column_docs": column_docs, "table_docs": table_docs}

Writing doc_builder.py


In [22]:
from doc_builder import build_documents

all_docs = {}
for db_id, kb in all_kbs.items():
    all_docs[db_id] = build_documents(db_id, kb)

total_col_docs = sum(len(d["column_docs"]) for d in all_docs.values())
total_table_docs = sum(len(d["table_docs"]) for d in all_docs.values())
print(f"Total column documents: {total_col_docs}")
print(f"Total table documents: {total_table_docs}")

print("\n--- Example column doc ---")
print(all_docs["student_club"]["column_docs"][2]["text"])

print("\n--- Example table doc ---")
print(all_docs["student_club"]["table_docs"][0]["text"])

Total column documents: 806
Total table documents: 79

--- Example column doc ---
Table: event | Column: event_date | Description: The date the event took place or is scheduled to take place | Value meaning: e.g. 2020-03-10T12:00:00 | Example values: 2020-03-10T12:00:00, 2020-01-14T09:30:00, 2019-11-24T09:00:00

--- Example table doc ---
Table: event | Columns: event_id, event_name, event_date, type, notes, location, status | Relationships: attendance.link_to_event -> event.event_id; budget.link_to_event -> event.event_id


In [23]:
%%writefile retriever.py
"""
Schema-aware retriever. Given a natural-language question and a database's
document set (from doc_builder.py), returns the tables/columns most likely
relevant -- using hybrid search (BM25 + dense embeddings), not just one method
alone, since keyword and semantic matching catch different failure modes:
BM25 catches exact column-name mentions, embeddings catch paraphrases.
"""

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


_embedder = None

def get_embedder():
    """Lazy-load so importing this module doesn't force a model download."""
    global _embedder
    if _embedder is None:
        # small, fast, good enough for schema text -- not a place to spend your GPU budget
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedder


def tokenize(text: str) -> list:
    return text.lower().replace("|", " ").replace(":", " ").split()


class SchemaRetriever:
    def __init__(self, column_docs: list):
        """column_docs: list of {doc_id, text, table, column, ...} from doc_builder.py"""
        self.docs = column_docs
        self.texts = [d["text"] for d in column_docs]

        # BM25 index
        tokenized = [tokenize(t) for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)

        # Dense embeddings, computed once and cached
        self.embeddings = get_embedder().encode(self.texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize for cosine similarity via dot product
        self.embeddings = self.embeddings / np.linalg.norm(self.embeddings, axis=1, keepdims=True)

    def bm25_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by BM25 score."""
        scores = self.bm25.get_scores(tokenize(query))
        return list(np.argsort(-scores))

    def dense_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by embedding cosine similarity."""
        q_emb = get_embedder().encode([query], convert_to_numpy=True)[0]
        q_emb = q_emb / np.linalg.norm(q_emb)
        sims = self.embeddings @ q_emb
        return list(np.argsort(-sims))

    def hybrid_rank(self, query: str, k_rrf: int = 60) -> list:
        """Reciprocal Rank Fusion: combine BM25 and dense rankings without
        needing to normalize/compare raw scores across the two methods
        (which are on incomparable scales). Each doc's fused score is the
        sum of 1/(k + rank) across both rankings -- a doc ranked highly by
        EITHER method gets a boost, which is exactly the point of hybrid search.
        Returns (doc_index, fused_score) sorted best-first.
        """
        bm25_order = self.bm25_rank(query)
        dense_order = self.dense_rank(query)

        fused_scores = {}
        for rank, idx in enumerate(bm25_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)
        for rank, idx in enumerate(dense_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)

        ranked = sorted(fused_scores.items(), key=lambda x: -x[1])
        return ranked

    def retrieve_columns(self, query: str, top_k: int = 15) -> list:
        """Returns top_k column doc dicts, best-first, each with a 'score' field."""
        ranked = self.hybrid_rank(query)[:top_k]
        results = []
        for idx, score in ranked:
            doc = dict(self.docs[idx])
            doc["score"] = score
            results.append(doc)
        return results

Writing retriever.py


In [24]:
from retriever import SchemaRetriever
from schema_kb import build_schema_kb
from doc_builder import build_documents

kb = build_schema_kb(DB_ROOT, "california_schools")
docs = build_documents("california_schools", kb)

retriever = SchemaRetriever(docs["column_docs"])

question = "What is the highest eligible free rate for K-12 students in the schools in Alameda County?"
results = retriever.retrieve_columns(question, top_k=8)

print(f"Question: {question}\n")
for r in results:
    print(f"  score={r['score']:.4f}  {r['table']}.{r['column']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Question: What is the highest eligible free rate for K-12 students in the schools in Alameda County?

  score=0.0323  schools.GSserved
  score=0.0315  schools.GSoffered
  score=0.0315  frpm.FRPM Count (K-12)
  score=0.0313  frpm.Free Meal Count (K-12)
  score=0.0308  schools.District
  score=0.0300  frpm.Free Meal Count (Ages 5-17)
  score=0.0299  frpm.Percent (%) Eligible Free (K-12)
  score=0.0288  frpm.Percent (%) Eligible Free (Ages 5-17)


In [25]:
print("=== BM25-only top 8 ===")
bm25_order = retriever.bm25_rank(question)[:8]
for idx in bm25_order:
    d = retriever.docs[idx]
    print(f"  {d['table']}.{d['column']}")

print("\n=== Dense-only top 8 ===")
dense_order = retriever.dense_rank(question)[:8]
for idx in dense_order:
    d = retriever.docs[idx]
    print(f"  {d['table']}.{d['column']}")

# where do the two "missing" gold columns actually rank?
target_cols = ["Enrollment (K-12)", "County Name"]
for target in target_cols:
    for rank_name, order in [("BM25", bm25_order), ("Dense", dense_order)]:
        full_order = retriever.bm25_rank(question) if rank_name == "BM25" else retriever.dense_rank(question)
        for rank, idx in enumerate(full_order):
            if retriever.docs[idx]["column"] == target:
                print(f"{target} -- {rank_name} rank: {rank}")
                break

=== BM25-only top 8 ===
  frpm.Free Meal Count (K-12)
  frpm.Free Meal Count (Ages 5-17)
  schools.GSserved
  frpm.FRPM Count (K-12)
  schools.GSoffered
  schools.District
  schools.SOCType
  schools.Virtual

=== Dense-only top 8 ===
  schools.GSserved
  schools.GSoffered
  frpm.FRPM Count (K-12)
  schools.District
  frpm.Percent (%) Eligible Free (K-12)
  schools.SOC
  frpm.Free Meal Count (K-12)
  schools.County
Enrollment (K-12) -- BM25 rank: 35
Enrollment (K-12) -- Dense rank: 13
County Name -- BM25 rank: 86
County Name -- Dense rank: 46


In [26]:
evidence = "Eligible free rate for K-12 = `Free Meal Count (K-12)` / `Enrollment (K-12)`"
combined_query = question + " " + evidence

results_with_evidence = retriever.retrieve_columns(combined_query, top_k=8)

print("=== Top 8 WITH evidence included in the query ===")
for r in results_with_evidence:
    print(f"  score={r['score']:.4f}  {r['table']}.{r['column']}")

=== Top 8 WITH evidence included in the query ===
  score=0.0328  frpm.Free Meal Count (K-12)
  score=0.0320  frpm.FRPM Count (K-12)
  score=0.0315  frpm.Free Meal Count (Ages 5-17)
  score=0.0312  frpm.Percent (%) Eligible Free (K-12)
  score=0.0303  schools.GSoffered
  score=0.0302  schools.GSserved
  score=0.0292  schools.District
  score=0.0291  frpm.Percent (%) Eligible Free (Ages 5-17)


In [27]:
%%writefile retriever.py
"""
Schema-aware retriever. Given a natural-language question and a database's
document set (from doc_builder.py), returns the tables/columns most likely
relevant -- using hybrid search (BM25 + dense embeddings), not just one method
alone, since keyword and semantic matching catch different failure modes:
BM25 catches exact column-name mentions, embeddings catch paraphrases.
"""

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


_embedder = None

def get_embedder():
    """Lazy-load so importing this module doesn't force a model download."""
    global _embedder
    if _embedder is None:
        # small, fast, good enough for schema text -- not a place to spend your GPU budget
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedder


import re

def tokenize(text: str) -> list:
    """Lowercase and strip punctuation (backticks, parens, commas, etc.) so
    tokens match cleanly. We deliberately keep hyphens inside words (e.g.
    'k-12' stays one token) since splitting it into 'k' and '12' would lose
    a meaningful unit -- but strip surrounding punctuation like backticks,
    parens, and equals signs that add noise without meaning."""
    text = text.lower()
    text = re.sub(r"[`(),=/|:]", " ", text)
    return text.split()


class SchemaRetriever:
    def __init__(self, column_docs: list):
        """column_docs: list of {doc_id, text, table, column, ...} from doc_builder.py"""
        self.docs = column_docs
        self.texts = [d["text"] for d in column_docs]

        # BM25 index
        tokenized = [tokenize(t) for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)

        # Dense embeddings, computed once and cached
        self.embeddings = get_embedder().encode(self.texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize for cosine similarity via dot product
        self.embeddings = self.embeddings / np.linalg.norm(self.embeddings, axis=1, keepdims=True)

    def bm25_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by BM25 score."""
        scores = self.bm25.get_scores(tokenize(query))
        return list(np.argsort(-scores))

    def dense_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by embedding cosine similarity."""
        q_emb = get_embedder().encode([query], convert_to_numpy=True)[0]
        q_emb = q_emb / np.linalg.norm(q_emb)
        sims = self.embeddings @ q_emb
        return list(np.argsort(-sims))

    def hybrid_rank(self, query: str, k_rrf: int = 60) -> list:
        """Reciprocal Rank Fusion: combine BM25 and dense rankings without
        needing to normalize/compare raw scores across the two methods
        (which are on incomparable scales). Each doc's fused score is the
        sum of 1/(k + rank) across both rankings -- a doc ranked highly by
        EITHER method gets a boost, which is exactly the point of hybrid search.
        Returns (doc_index, fused_score) sorted best-first.
        """
        bm25_order = self.bm25_rank(query)
        dense_order = self.dense_rank(query)

        fused_scores = {}
        for rank, idx in enumerate(bm25_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)
        for rank, idx in enumerate(dense_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)

        ranked = sorted(fused_scores.items(), key=lambda x: -x[1])
        return ranked

    def retrieve_columns(self, query: str, top_k: int = 15) -> list:
        """Returns top_k column doc dicts, best-first, each with a 'score' field."""
        ranked = self.hybrid_rank(query)[:top_k]
        results = []
        for idx, score in ranked:
            doc = dict(self.docs[idx])
            doc["score"] = score
            results.append(doc)
        return results

Overwriting retriever.py


In [28]:
# reload the module fresh since we edited the file
import importlib
import retriever as retriever_module
importlib.reload(retriever_module)
from retriever import SchemaRetriever

retriever = SchemaRetriever(docs["column_docs"])  # rebuild -- old BM25 index used the buggy tokenizer

results = retriever.retrieve_columns(combined_query, top_k=8)
print("=== Top 8 WITH fixed tokenizer ===")
for r in results:
    print(f"  score={r['score']:.4f}  {r['table']}.{r['column']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== Top 8 WITH fixed tokenizer ===
  score=0.0328  frpm.Free Meal Count (K-12)
  score=0.0323  frpm.FRPM Count (K-12)
  score=0.0313  frpm.Free Meal Count (Ages 5-17)
  score=0.0312  frpm.Percent (%) Eligible Free (K-12)
  score=0.0306  schools.GSserved
  score=0.0301  frpm.Enrollment (K-12)
  score=0.0301  frpm.Percent (%) Eligible FRPM (K-12)
  score=0.0296  schools.GSoffered


In [29]:
%%writefile schema_kb.py
"""
Builds a structured knowledge base for one database: every column with its
description, every table, and the PK/FK relationships between them. This is
the raw material the retriever will search over -- get this right first,
everything downstream depends on it.
"""

import sqlite3
import pandas as pd
import os


def get_foreign_keys(db_path: str) -> list:
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    fks = []
    for t in tables:
        cur.execute(f"PRAGMA foreign_key_list(`{t}`)")
        for row in cur.fetchall():
            fks.append({
                "from_table": t,
                "from_col": row[3],
                "to_table": row[2],
                "to_col": row[4],
            })
    conn.close()
    return fks


def get_primary_keys(db_path: str) -> dict:
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    pks = {}
    for t in tables:
        cur.execute(f"PRAGMA table_info(`{t}`)")
        pk_cols = [row[1] for row in cur.fetchall() if row[5] > 0]
        pks[t] = pk_cols
    conn.close()
    return pks


def get_sample_values(db_path: str, table: str, column: str, n: int = 3) -> list:
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
        cur = conn.cursor()
        cur.execute(f"SELECT DISTINCT `{column}` FROM `{table}` WHERE `{column}` IS NOT NULL LIMIT {n}")
        vals = [row[0] for row in cur.fetchall()]
        conn.close()
        return vals
    except Exception:
        return []


def search_value_in_db(db_path: str, candidate: str, max_hits: int = 5) -> list:
    """Search every TEXT column in the database for an exact (case-insensitive)
    match of `candidate`. Returns [{table, column, matched_value}, ...].

    This is deliberately a targeted search, not a reuse of the small random
    sample_values -- a value like 'Alameda' has no reliable chance of showing
    up in a 3-value random sample out of ~58 counties, so we search directly.

    Cost note: this runs one query per TEXT column per candidate word, so it's
    fine for a handful of question-words at inference time, but NOT something
    to run per-training-example at scale without caching -- see retriever.py's
    caller for how we limit which words we search.
    """
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    hits = []
    for t in tables:
        cur.execute(f"PRAGMA table_info(`{t}`)")
        cols = cur.fetchall()
        for col_row in cols:
            col_name, col_type = col_row[1], col_row[2]
            if col_type and "CHAR" not in col_type.upper() and "TEXT" not in col_type.upper():
                continue
            try:
                cur.execute(
                    f"SELECT DISTINCT `{col_name}` FROM `{t}` "
                    f"WHERE `{col_name}` = ? COLLATE NOCASE LIMIT 1",
                    (candidate,)
                )
                row = cur.fetchone()
                if row:
                    hits.append({"table": t, "column": col_name, "matched_value": row[0]})
                    if len(hits) >= max_hits:
                        conn.close()
                        return hits
            except Exception:
                continue
    conn.close()
    return hits


def find_value_matches(db_path: str, question: str, min_word_len: int = 4) -> list:
    """Extracts candidate proper-noun-like words from the question (crude but
    effective: capitalized words, length >= min_word_len, skip common question
    words) and searches for each as an exact value in the DB.
    Returns a flat list of {table, column, matched_value, matched_word}.
    """
    import re
    words = re.findall(r"[A-Za-z][A-Za-z\-']+", question)
    skip = {"What", "Which", "How", "Where", "When", "Who", "The", "For", "In", "Of", "Is", "Are"}
    candidates = [w for w in words if len(w) >= min_word_len and w[0].isupper() and w not in skip]

    matches = []
    for word in candidates:
        hits = search_value_in_db(db_path, word)
        for h in hits:
            h["matched_word"] = word
            matches.append(h)
    return matches


def load_descriptions(desc_dir: str) -> dict:
    tables = {}
    for fname in os.listdir(desc_dir):
        if not fname.endswith(".csv"):
            continue
        table_name = fname[:-4]
        path = os.path.join(desc_dir, fname)
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="latin-1")
        tables[table_name] = df
    return tables


def build_schema_kb(db_root: str, db_id: str) -> dict:
    db_path = f"{db_root}/{db_id}/{db_id}.sqlite"
    desc_dir = f"{db_root}/{db_id}/database_description"

    fks = get_foreign_keys(db_path)
    pks = get_primary_keys(db_path)
    descriptions = load_descriptions(desc_dir) if os.path.exists(desc_dir) else {}

    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]

    columns = []
    for t in tables:
        cur.execute(f"PRAGMA table_info(`{t}`)")
        table_cols = cur.fetchall()

        desc_df = descriptions.get(t)

        for col_row in table_cols:
            col_name = col_row[1]
            col_type = col_row[2]

            description = ""
            value_description = ""
            if desc_df is not None:
                match = desc_df[desc_df["original_column_name"].astype(str).str.strip() == col_name]
                if len(match) > 0:
                    row = match.iloc[0]
                    description = str(row.get("column_description", "") or "")
                    value_description = str(row.get("value_description", "") or "")
                    if description == "nan":
                        description = ""
                    if value_description == "nan":
                        value_description = ""

            columns.append({
                "table": t,
                "column": col_name,
                "type": col_type,
                "description": description,
                "value_description": value_description,
                "is_primary_key": col_name in pks.get(t, []),
                "sample_values": get_sample_values(db_path, t, col_name),
            })

    conn.close()
    return {"tables": tables, "columns": columns, "foreign_keys": fks}

Overwriting schema_kb.py


In [30]:
import importlib
import schema_kb
importlib.reload(schema_kb)
from schema_kb import find_value_matches

db_path = f"{DB_ROOT}/california_schools/california_schools.sqlite"
matches = find_value_matches(db_path, question)

print(f"Question: {question}\n")
print("Value matches found:")
for m in matches:
    print(f"  '{m['matched_word']}' -> {m['table']}.{m['column']} (value: '{m['matched_value']}')")

Question: What is the highest eligible free rate for K-12 students in the schools in Alameda County?

Value matches found:
  'Alameda' -> frpm.County Name (value: 'Alameda')
  'Alameda' -> satscores.cname (value: 'Alameda')
  'Alameda' -> schools.County (value: 'Alameda')
  'Alameda' -> schools.City (value: 'Alameda')
  'Alameda' -> schools.MailCity (value: 'Alameda')


In [31]:
%%writefile retriever.py
"""
Schema-aware retriever. Given a natural-language question and a database's
document set (from doc_builder.py), returns the tables/columns most likely
relevant -- using hybrid search (BM25 + dense embeddings), not just one method
alone, since keyword and semantic matching catch different failure modes:
BM25 catches exact column-name mentions, embeddings catch paraphrases.
"""

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


_embedder = None

def get_embedder():
    """Lazy-load so importing this module doesn't force a model download."""
    global _embedder
    if _embedder is None:
        # small, fast, good enough for schema text -- not a place to spend your GPU budget
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedder


import re

def tokenize(text: str) -> list:
    """Lowercase and strip punctuation (backticks, parens, commas, etc.) so
    tokens match cleanly. We deliberately keep hyphens inside words (e.g.
    'k-12' stays one token) since splitting it into 'k' and '12' would lose
    a meaningful unit -- but strip surrounding punctuation like backticks,
    parens, and equals signs that add noise without meaning."""
    text = text.lower()
    text = re.sub(r"[`(),=/|:]", " ", text)
    return text.split()


class SchemaRetriever:
    def __init__(self, column_docs: list):
        """column_docs: list of {doc_id, text, table, column, ...} from doc_builder.py"""
        self.docs = column_docs
        self.texts = [d["text"] for d in column_docs]

        # BM25 index
        tokenized = [tokenize(t) for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)

        # Dense embeddings, computed once and cached
        self.embeddings = get_embedder().encode(self.texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize for cosine similarity via dot product
        self.embeddings = self.embeddings / np.linalg.norm(self.embeddings, axis=1, keepdims=True)

    def bm25_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by BM25 score."""
        scores = self.bm25.get_scores(tokenize(query))
        return list(np.argsort(-scores))

    def dense_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by embedding cosine similarity."""
        q_emb = get_embedder().encode([query], convert_to_numpy=True)[0]
        q_emb = q_emb / np.linalg.norm(q_emb)
        sims = self.embeddings @ q_emb
        return list(np.argsort(-sims))

    def hybrid_rank(self, query: str, k_rrf: int = 60) -> list:
        """Reciprocal Rank Fusion: combine BM25 and dense rankings without
        needing to normalize/compare raw scores across the two methods
        (which are on incomparable scales). Each doc's fused score is the
        sum of 1/(k + rank) across both rankings -- a doc ranked highly by
        EITHER method gets a boost, which is exactly the point of hybrid search.
        Returns (doc_index, fused_score) sorted best-first.
        """
        bm25_order = self.bm25_rank(query)
        dense_order = self.dense_rank(query)

        fused_scores = {}
        for rank, idx in enumerate(bm25_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)
        for rank, idx in enumerate(dense_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)

        ranked = sorted(fused_scores.items(), key=lambda x: -x[1])
        return ranked

    def retrieve_columns(self, query: str, top_k: int = 15, value_matches: list = None,
                          value_boost: float = 0.05) -> list:
        """Returns top_k column doc dicts, best-first, each with a 'score' field.

        value_matches: optional output from schema_kb.find_value_matches(), i.e.
        [{table, column, matched_value, matched_word}, ...]. Columns matching a
        real cell value (e.g. 'Alameda' found inside frpm.County Name) get a
        flat score boost. This is deliberately additive, not a hard override --
        BM25/dense already found some of these; value matching's job is to pull
        in cases neither text-based method could catch, since the matched word
        (e.g. a proper noun like 'Alameda') often doesn't appear in the column's
        name or description at all, only in the data itself.
        """
        ranked = self.hybrid_rank(query)
        fused_scores = dict(ranked)

        if value_matches:
            # build a lookup: (table, column) -> already in this db's docs?
            doc_lookup = {(d["table"], d["column"]): idx for idx, d in enumerate(self.docs)}
            for vm in value_matches:
                key = (vm["table"], vm["column"])
                if key in doc_lookup:
                    idx = doc_lookup[key]
                    fused_scores[idx] = fused_scores.get(idx, 0) + value_boost

        ranked_final = sorted(fused_scores.items(), key=lambda x: -x[1])[:top_k]
        results = []
        for idx, score in ranked_final:
            doc = dict(self.docs[idx])
            doc["score"] = score
            results.append(doc)
        return results

Overwriting retriever.py


In [32]:
import importlib
import retriever as retriever_module
importlib.reload(retriever_module)
from retriever import SchemaRetriever
from schema_kb import find_value_matches

retriever = SchemaRetriever(docs["column_docs"])  # rebuild fresh with the updated class

db_path = f"{DB_ROOT}/california_schools/california_schools.sqlite"
value_matches = find_value_matches(db_path, question)

results = retriever.retrieve_columns(combined_query, top_k=8, value_matches=value_matches)

print("=== Top 8, full pipeline: BM25 + dense + evidence + value-matching ===")
for r in results:
    print(f"  score={r['score']:.4f}  {r['table']}.{r['column']}")

# check: are all 3 gold columns now present?
gold_cols = {"Free Meal Count (K-12)", "Enrollment (K-12)", "County Name"}
found_cols = {r["column"] for r in results}
print(f"\nGold columns found: {gold_cols & found_cols}")
print(f"Gold columns MISSING: {gold_cols - found_cols}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== Top 8, full pipeline: BM25 + dense + evidence + value-matching ===
  score=0.0760  schools.County
  score=0.0703  frpm.County Name
  score=0.0694  schools.City
  score=0.0686  schools.MailCity
  score=0.0672  satscores.cname
  score=0.0328  frpm.Free Meal Count (K-12)
  score=0.0323  frpm.FRPM Count (K-12)
  score=0.0313  frpm.Free Meal Count (Ages 5-17)

Gold columns found: {'Free Meal Count (K-12)', 'County Name'}
Gold columns MISSING: {'Enrollment (K-12)'}


In [33]:
%%writefile retriever.py
"""
Schema-aware retriever. Given a natural-language question and a database's
document set (from doc_builder.py), returns the tables/columns most likely
relevant -- using hybrid search (BM25 + dense embeddings), not just one method
alone, since keyword and semantic matching catch different failure modes:
BM25 catches exact column-name mentions, embeddings catch paraphrases.
"""

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


_embedder = None

def get_embedder():
    """Lazy-load so importing this module doesn't force a model download."""
    global _embedder
    if _embedder is None:
        # small, fast, good enough for schema text -- not a place to spend your GPU budget
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    return _embedder


import re

def tokenize(text: str) -> list:
    """Lowercase and strip punctuation (backticks, parens, commas, etc.) so
    tokens match cleanly. We deliberately keep hyphens inside words (e.g.
    'k-12' stays one token) since splitting it into 'k' and '12' would lose
    a meaningful unit -- but strip surrounding punctuation like backticks,
    parens, and equals signs that add noise without meaning."""
    text = text.lower()
    text = re.sub(r"[`(),=/|:]", " ", text)
    return text.split()


class SchemaRetriever:
    def __init__(self, column_docs: list):
        """column_docs: list of {doc_id, text, table, column, ...} from doc_builder.py"""
        self.docs = column_docs
        self.texts = [d["text"] for d in column_docs]

        # BM25 index
        tokenized = [tokenize(t) for t in self.texts]
        self.bm25 = BM25Okapi(tokenized)

        # Dense embeddings, computed once and cached
        self.embeddings = get_embedder().encode(self.texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize for cosine similarity via dot product
        self.embeddings = self.embeddings / np.linalg.norm(self.embeddings, axis=1, keepdims=True)

    def bm25_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by BM25 score."""
        scores = self.bm25.get_scores(tokenize(query))
        return list(np.argsort(-scores))

    def dense_rank(self, query: str) -> list:
        """Returns doc indices ranked best-first by embedding cosine similarity."""
        q_emb = get_embedder().encode([query], convert_to_numpy=True)[0]
        q_emb = q_emb / np.linalg.norm(q_emb)
        sims = self.embeddings @ q_emb
        return list(np.argsort(-sims))

    def hybrid_rank(self, query: str, k_rrf: int = 60) -> list:
        """Reciprocal Rank Fusion: combine BM25 and dense rankings without
        needing to normalize/compare raw scores across the two methods
        (which are on incomparable scales). Each doc's fused score is the
        sum of 1/(k + rank) across both rankings -- a doc ranked highly by
        EITHER method gets a boost, which is exactly the point of hybrid search.
        Returns (doc_index, fused_score) sorted best-first.
        """
        bm25_order = self.bm25_rank(query)
        dense_order = self.dense_rank(query)

        fused_scores = {}
        for rank, idx in enumerate(bm25_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)
        for rank, idx in enumerate(dense_order):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (k_rrf + rank + 1)

        ranked = sorted(fused_scores.items(), key=lambda x: -x[1])
        return ranked

    def retrieve_columns(self, query: str, top_k: int = 15, value_matches: list = None) -> list:
        """Returns column doc dicts, each with a 'score' and 'source' field.

        Design: text-based ranking (BM25+dense fusion) and value-matching are
        UNIONED, not blended into one competing score. Blending them (an
        earlier version of this code) caused a real bug: value matches for an
        ambiguous word like 'Alameda' (5 columns contain that literal value)
        scored so consistently high they filled the entire top_k budget and
        pushed out a column (Enrollment (K-12)) that only text/semantic
        matching could find. Keeping the two candidate pools separate means
        one strong signal can never silently starve the other.
        """
        ranked = self.hybrid_rank(query)[:top_k]
        results = []
        seen = set()
        for idx, score in ranked:
            doc = dict(self.docs[idx])
            doc["score"] = score
            doc["source"] = "text"
            results.append(doc)
            seen.add((doc["table"], doc["column"]))

        if value_matches:
            doc_lookup = {(d["table"], d["column"]): idx for idx, d in enumerate(self.docs)}
            for vm in value_matches:
                key = (vm["table"], vm["column"])
                if key in seen or key not in doc_lookup:
                    continue
                idx = doc_lookup[key]
                doc = dict(self.docs[idx])
                doc["score"] = None  # not from the ranked pool, no comparable score
                doc["source"] = "value_match"
                doc["matched_value"] = vm["matched_value"]
                doc["matched_word"] = vm["matched_word"]
                results.append(doc)
                seen.add(key)

        return results

Overwriting retriever.py


In [34]:
import importlib
import retriever as retriever_module
importlib.reload(retriever_module)
from retriever import SchemaRetriever

retriever = SchemaRetriever(docs["column_docs"])

results = retriever.retrieve_columns(combined_query, top_k=8, value_matches=value_matches)

print("=== Union-based results: top_k text-ranked + guaranteed value matches ===")
for r in results:
    score_str = f"{r['score']:.4f}" if r['score'] is not None else "  --  "
    print(f"  score={score_str}  source={r['source']:12s}  {r['table']}.{r['column']}")

gold_cols = {"Free Meal Count (K-12)", "Enrollment (K-12)", "County Name"}
found_cols = {r["column"] for r in results}
print(f"\nGold columns found: {gold_cols & found_cols}")
print(f"Gold columns MISSING: {gold_cols - found_cols}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== Union-based results: top_k text-ranked + guaranteed value matches ===
  score=0.0328  source=text          frpm.Free Meal Count (K-12)
  score=0.0323  source=text          frpm.FRPM Count (K-12)
  score=0.0313  source=text          frpm.Free Meal Count (Ages 5-17)
  score=0.0312  source=text          frpm.Percent (%) Eligible Free (K-12)
  score=0.0306  source=text          schools.GSserved
  score=0.0301  source=text          frpm.Enrollment (K-12)
  score=0.0301  source=text          frpm.Percent (%) Eligible FRPM (K-12)
  score=0.0296  source=text          schools.GSoffered
  score=  --    source=value_match   frpm.County Name
  score=  --    source=value_match   satscores.cname
  score=  --    source=value_match   schools.County
  score=  --    source=value_match   schools.City
  score=  --    source=value_match   schools.MailCity

Gold columns found: {'Enrollment (K-12)', 'Free Meal Count (K-12)', 'County Name'}
Gold columns MISSING: set()


In [35]:
from tqdm.notebook import tqdm
import sqlglot
from sqlglot import exp

def extract_gold_columns(sql: str, db_id: str) -> set:
    """Parse gold SQL to get the (table, column) pairs actually used --
    this is our ground truth for measuring retrieval recall."""
    try:
        parsed = sqlglot.parse_one(sql, read="sqlite")
        cols = set()
        for col in parsed.find_all(exp.Column):
            table = col.table if col.table else None
            cols.add((table, col.name))
        return cols
    except Exception:
        return set()

retrieval_results = []

for ex in tqdm(sample, desc="Retrieving"):
    db_id = ex["db_id"]
    if db_id not in all_docs:
        continue

    r = SchemaRetriever(all_docs[db_id]["column_docs"])
    q_combined = ex["question"] + " " + ex.get("evidence", "")

    db_path = f"{DB_ROOT}/{db_id}/{db_id}.sqlite"
    vmatches = find_value_matches(db_path, ex["question"])

    retrieved = r.retrieve_columns(q_combined, top_k=15, value_matches=vmatches)
    retrieved_cols = {(rc["table"], rc["column"]) for rc in retrieved}

    gold_cols_raw = extract_gold_columns(ex["SQL"], db_id)
    # gold columns from sqlglot don't always have table qualifiers -- match
    # loosely by column name for now, refine once we see real numbers
    gold_col_names = {c[1].lower() for c in gold_cols_raw}
    retrieved_col_names = {rc[1].lower() for rc in retrieved_cols}

    hit = gold_col_names.issubset(retrieved_col_names) if gold_col_names else None
    recall = len(gold_col_names & retrieved_col_names) / len(gold_col_names) if gold_col_names else None

    retrieval_results.append({
        "question_id": ex["question_id"],
        "db_id": db_id,
        "difficulty": ex["difficulty"],
        "gold_cols": list(gold_col_names),
        "retrieved_cols": list(retrieved_col_names),
        "full_coverage": hit,
        "recall": recall,
    })

valid_results = [r for r in retrieval_results if r["recall"] is not None]
avg_recall = sum(r["recall"] for r in valid_results) / len(valid_results)
full_coverage_rate = sum(r["full_coverage"] for r in valid_results) / len(valid_results)

print(f"Questions scored: {len(valid_results)}")
print(f"Average column recall: {avg_recall:.3f}")
print(f"Full-coverage rate (ALL gold columns retrieved): {full_coverage_rate:.3f}")

import json
with open("/kaggle/working/retrieval_results_200.json", "w") as f:
    json.dump(retrieval_results, f, indent=2)

Retrieving:   0%|          | 0/200 [00:00<?, ?it/s]

Questions scored: 200
Average column recall: 0.813
Full-coverage rate (ALL gold columns retrieved): 0.520


In [36]:
import json
with open("/kaggle/working/retrieval_metrics_summary.json", "w") as f:
    json.dump({
        "avg_column_recall": avg_recall,
        "full_coverage_rate": full_coverage_rate,
        "n_questions": len(valid_results),
        "known_limitation": "recall computed on unqualified column names; may overestimate when duplicate column names exist across tables"
    }, f, indent=2)

In [37]:
%%writefile context_builder.py
"""
Turns retrieved columns into an actual prompt-ready schema context.

Three steps, each solving a distinct problem:
1. Aggregate columns -> tables (a table is "relevant" if any of its columns
   were retrieved; value-matched columns count as strong signal).
2. FK-graph expansion: if the question needs columns from two tables that
   aren't directly connected, find the bridge table(s) needed to JOIN them.
   Without this, the model gets fk-disconnected table fragments and can't
   write a valid join.
3. Render as compact text: full column list per selected table (not just
   the retrieved ones -- a model needs to see a table's real structure to
   use it correctly), with descriptions inline, plus a relationships section.
"""

from collections import deque


def aggregate_columns_to_tables(retrieved_columns: list, top_m_tables: int = 6) -> list:
    """Returns table names ranked by relevance, best-first.
    value_match columns (score=None) are treated as maximally relevant --
    they came from an exact match to real data, which is a strong signal
    even though it's not comparable to the BM25/dense fused score scale.
    """
    VALUE_MATCH_SCORE = 999.0  # sentinel: always outranks text-based scores

    table_scores = {}
    for col in retrieved_columns:
        score = col["score"] if col["score"] is not None else VALUE_MATCH_SCORE
        t = col["table"]
        table_scores[t] = max(table_scores.get(t, -1), score)

    ranked = sorted(table_scores.items(), key=lambda x: -x[1])
    return [t for t, _ in ranked[:top_m_tables]]


def build_fk_graph(foreign_keys: list) -> dict:
    """Undirected adjacency list over tables, built from FK relationships."""
    graph = {}
    for fk in foreign_keys:
        a, b = fk["from_table"], fk["to_table"]
        graph.setdefault(a, set()).add(b)
        graph.setdefault(b, set()).add(a)
    return graph


def shortest_path(graph: dict, start: str, end: str) -> list:
    """BFS shortest path between two tables in the FK graph. Returns the
    list of tables on the path (inclusive of start/end), or [] if unreachable."""
    if start == end:
        return [start]
    if start not in graph or end not in graph:
        return []

    visited = {start}
    queue = deque([[start]])
    while queue:
        path = queue.popleft()
        node = path[-1]
        for neighbor in graph.get(node, []):
            if neighbor == end:
                return path + [neighbor]
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])
    return []  # no path found -- tables are in disconnected components


def expand_with_fk_bridges(selected_tables: list, foreign_keys: list) -> list:
    """Given a set of relevant tables, add any bridge tables needed to
    connect them all via foreign keys, so the model can actually write
    JOINs between them. Uses pairwise shortest-path and unions the results
    -- not a true minimal Steiner tree, but good enough for schemas this
    size, and simple enough to reason about and debug.
    """
    graph = build_fk_graph(foreign_keys)
    expanded = set(selected_tables)

    for i, t1 in enumerate(selected_tables):
        for t2 in selected_tables[i + 1:]:
            path = shortest_path(graph, t1, t2)
            expanded.update(path)  # empty list if unreachable -- fine, just adds nothing

    return list(expanded)


def render_schema_context(kb: dict, selected_tables: list) -> str:
    """Renders the final compact schema text for the prompt: one CREATE
    TABLE-style block per selected table, full column list (not just
    retrieved columns -- the model needs the whole table shape to use it
    correctly), with descriptions inline, plus a relationships section.
    """
    lines = []

    cols_by_table = {}
    for col in kb["columns"]:
        cols_by_table.setdefault(col["table"], []).append(col)

    for t in selected_tables:
        if t not in cols_by_table:
            continue  # defensive: shouldn't happen, but don't crash the prompt build
        lines.append(f"CREATE TABLE {t} (")
        col_lines = []
        for col in cols_by_table[t]:
            piece = f"  {col['column']} {col['type']}"
            if col["is_primary_key"]:
                piece += " PRIMARY KEY"
            if col["description"]:
                piece += f"  -- {col['description']}"
            col_lines.append(piece)
        lines.append(",\n".join(col_lines))
        lines.append(");\n")

    relevant_fks = [
        fk for fk in kb["foreign_keys"]
        if fk["from_table"] in selected_tables and fk["to_table"] in selected_tables
    ]
    if relevant_fks:
        lines.append("-- Relationships:")
        for fk in relevant_fks:
            lines.append(f"-- {fk['from_table']}.{fk['from_col']} = {fk['to_table']}.{fk['to_col']}")

    return "\n".join(lines)


def build_retrieved_context(kb: dict, retrieved_columns: list, top_m_tables: int = 6) -> str:
    """Full pipeline: retrieved columns -> relevant tables -> FK-expanded ->
    rendered schema text. This is the single function the generation step calls."""
    selected = aggregate_columns_to_tables(retrieved_columns, top_m_tables)
    expanded = expand_with_fk_bridges(selected, kb["foreign_keys"])
    return render_schema_context(kb, expanded)

Writing context_builder.py


In [38]:
from context_builder import build_retrieved_context

# use the results from your last full-pipeline test (text + value matches)
schema_context = build_retrieved_context(kb, results, top_m_tables=6)

print(schema_context)
print(f"\n--- Context length: {len(schema_context)} chars ---")

CREATE TABLE frpm (
  CDSCode TEXT PRIMARY KEY  -- CDSCode,
  Academic Year TEXT  -- Academic Year,
  County Code TEXT  -- County Code,
  District Code INTEGER  -- District Code,
  School Code TEXT  -- School Code,
  County Name TEXT  -- County Code ,
  District Name TEXT  -- District Name ,
  School Name TEXT  -- School Name ,
  District Type TEXT  -- District Type,
  School Type TEXT  -- School Type ,
  Educational Option Type TEXT  -- Educational Option Type,
  NSLP Provision Status TEXT  -- NSLP Provision Status,
  Charter School (Y/N) INTEGER  -- Charter School (Y/N),
  Charter School Number TEXT  -- Charter School Number,
  Charter Funding Type TEXT  -- Charter Funding Type,
  IRC INTEGER,
  Low Grade TEXT  -- Low Grade,
  High Grade TEXT  -- High Grade,
  Enrollment (K-12) REAL  -- Enrollment (K-12),
  Free Meal Count (K-12) REAL  -- Free Meal Count (K-12),
  Percent (%) Eligible Free (K-12) REAL,
  FRPM Count (K-12) REAL  -- Free or Reduced Price Meal Count (K-12),
  Percent (%

In [39]:
%%writefile context_builder.py
"""
Turns retrieved columns into an actual prompt-ready schema context.

Three steps, each solving a distinct problem:
1. Aggregate columns -> tables (a table is "relevant" if any of its columns
   were retrieved; value-matched columns count as strong signal).
2. FK-graph expansion: if the question needs columns from two tables that
   aren't directly connected, find the bridge table(s) needed to JOIN them.
   Without this, the model gets fk-disconnected table fragments and can't
   write a valid join.
3. Render as compact text: full column list per selected table (not just
   the retrieved ones -- a model needs to see a table's real structure to
   use it correctly), with descriptions inline, plus a relationships section.
"""

from collections import deque


def aggregate_columns_to_tables(retrieved_columns: list, top_m_tables: int = 6) -> list:
    """Returns table names ranked by relevance, best-first.
    value_match columns (score=None) are treated as maximally relevant --
    they came from an exact match to real data, which is a strong signal
    even though it's not comparable to the BM25/dense fused score scale.
    """
    VALUE_MATCH_SCORE = 999.0  # sentinel: always outranks text-based scores

    table_scores = {}
    for col in retrieved_columns:
        score = col["score"] if col["score"] is not None else VALUE_MATCH_SCORE
        t = col["table"]
        table_scores[t] = max(table_scores.get(t, -1), score)

    ranked = sorted(table_scores.items(), key=lambda x: -x[1])
    return [t for t, _ in ranked[:top_m_tables]]


def build_fk_graph(foreign_keys: list) -> dict:
    """Undirected adjacency list over tables, built from FK relationships."""
    graph = {}
    for fk in foreign_keys:
        a, b = fk["from_table"], fk["to_table"]
        graph.setdefault(a, set()).add(b)
        graph.setdefault(b, set()).add(a)
    return graph


def shortest_path(graph: dict, start: str, end: str) -> list:
    """BFS shortest path between two tables in the FK graph. Returns the
    list of tables on the path (inclusive of start/end), or [] if unreachable."""
    if start == end:
        return [start]
    if start not in graph or end not in graph:
        return []

    visited = {start}
    queue = deque([[start]])
    while queue:
        path = queue.popleft()
        node = path[-1]
        for neighbor in graph.get(node, []):
            if neighbor == end:
                return path + [neighbor]
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])
    return []  # no path found -- tables are in disconnected components


def expand_with_fk_bridges(selected_tables: list, foreign_keys: list) -> list:
    """Given a set of relevant tables, add any bridge tables needed to
    connect them all via foreign keys, so the model can actually write
    JOINs between them. Uses pairwise shortest-path and unions the results
    -- not a true minimal Steiner tree, but good enough for schemas this
    size, and simple enough to reason about and debug.
    """
    graph = build_fk_graph(foreign_keys)
    expanded = set(selected_tables)

    for i, t1 in enumerate(selected_tables):
        for t2 in selected_tables[i + 1:]:
            path = shortest_path(graph, t1, t2)
            expanded.update(path)  # empty list if unreachable -- fine, just adds nothing

    return list(expanded)


def render_schema_context(kb: dict, selected_tables: list) -> str:
    """Renders the final compact schema text for the prompt: one CREATE
    TABLE-style block per selected table, full column list (not just
    retrieved columns -- the model needs the whole table shape to use it
    correctly), with descriptions inline, plus a relationships section.
    """
    lines = []

    cols_by_table = {}
    for col in kb["columns"]:
        cols_by_table.setdefault(col["table"], []).append(col)

    def quote_ident(name: str) -> str:
        """Backtick-quote any identifier with a space or special character,
        matching the convention BIRD's gold SQL actually uses (e.g.
        `Free Meal Count (K-12)`). A plain identifier like CDSCode is left
        unquoted since that matches gold SQL style too -- consistency with
        gold SQL's own quoting habits matters more than always/never quoting."""
        if name.isidentifier():
            return name
        return f"`{name}`"

    for t in selected_tables:
        if t not in cols_by_table:
            continue  # defensive: shouldn't happen, but don't crash the prompt build
        lines.append(f"CREATE TABLE {quote_ident(t)} (")
        n_cols = len(cols_by_table[t])
        for i, col in enumerate(cols_by_table[t]):
            is_last = (i == n_cols - 1)
            piece = f"  {quote_ident(col['column'])} {col['type']}"
            if col["is_primary_key"]:
                piece += " PRIMARY KEY"
            sep = "" if is_last else ","  # comma goes BEFORE the comment, not after
            if col["description"]:
                piece += f"{sep}  -- {col['description']}"
            else:
                piece += sep
            lines.append(piece)
        lines.append(");\n")

    relevant_fks = [
        fk for fk in kb["foreign_keys"]
        if fk["from_table"] in selected_tables and fk["to_table"] in selected_tables
    ]
    if relevant_fks:
        lines.append("-- Relationships:")
        for fk in relevant_fks:
            lines.append(f"-- {fk['from_table']}.{fk['from_col']} = {fk['to_table']}.{fk['to_col']}")

    return "\n".join(lines)


def build_retrieved_context(kb: dict, retrieved_columns: list, top_m_tables: int = 6) -> str:
    """Full pipeline: retrieved columns -> relevant tables -> FK-expanded ->
    rendered schema text. This is the single function the generation step calls."""
    selected = aggregate_columns_to_tables(retrieved_columns, top_m_tables)
    expanded = expand_with_fk_bridges(selected, kb["foreign_keys"])
    return render_schema_context(kb, expanded)

Overwriting context_builder.py


In [40]:
import pandas as pd
desc_path = f"{DB_ROOT}/california_schools/database_description/frpm.csv"
df = pd.read_csv(desc_path, encoding="utf-8-sig")
print(df[df["original_column_name"].astype(str).str.strip() == "County Name"])

  original_column_name  column_name column_description data_format  \
5          County Name          NaN       County Code         text   

  value_description  
5               NaN  


In [41]:
%%writefile context_builder.py
"""
Turns retrieved columns into an actual prompt-ready schema context.

Three steps, each solving a distinct problem:
1. Aggregate columns -> tables (a table is "relevant" if any of its columns
   were retrieved; value-matched columns count as strong signal).
2. FK-graph expansion: if the question needs columns from two tables that
   aren't directly connected, find the bridge table(s) needed to JOIN them.
   Without this, the model gets fk-disconnected table fragments and can't
   write a valid join.
3. Render as compact text: full column list per selected table (not just
   the retrieved ones -- a model needs to see a table's real structure to
   use it correctly), with descriptions inline, plus a relationships section.
"""

from collections import deque


def aggregate_columns_to_tables(retrieved_columns: list, top_m_tables: int = 6) -> list:
    """Returns table names ranked by relevance, best-first.
    value_match columns (score=None) are treated as maximally relevant --
    they came from an exact match to real data, which is a strong signal
    even though it's not comparable to the BM25/dense fused score scale.
    """
    VALUE_MATCH_SCORE = 999.0  # sentinel: always outranks text-based scores

    table_scores = {}
    for col in retrieved_columns:
        score = col["score"] if col["score"] is not None else VALUE_MATCH_SCORE
        t = col["table"]
        table_scores[t] = max(table_scores.get(t, -1), score)

    ranked = sorted(table_scores.items(), key=lambda x: -x[1])
    return [t for t, _ in ranked[:top_m_tables]]


def build_fk_graph(foreign_keys: list) -> dict:
    """Undirected adjacency list over tables, built from FK relationships."""
    graph = {}
    for fk in foreign_keys:
        a, b = fk["from_table"], fk["to_table"]
        graph.setdefault(a, set()).add(b)
        graph.setdefault(b, set()).add(a)
    return graph


def shortest_path(graph: dict, start: str, end: str) -> list:
    """BFS shortest path between two tables in the FK graph. Returns the
    list of tables on the path (inclusive of start/end), or [] if unreachable."""
    if start == end:
        return [start]
    if start not in graph or end not in graph:
        return []

    visited = {start}
    queue = deque([[start]])
    while queue:
        path = queue.popleft()
        node = path[-1]
        for neighbor in graph.get(node, []):
            if neighbor == end:
                return path + [neighbor]
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])
    return []  # no path found -- tables are in disconnected components


def expand_with_fk_bridges(selected_tables: list, foreign_keys: list) -> list:
    """Given a set of relevant tables, add any bridge tables needed to
    connect them all via foreign keys, so the model can actually write
    JOINs between them. Uses pairwise shortest-path and unions the results
    -- not a true minimal Steiner tree, but good enough for schemas this
    size, and simple enough to reason about and debug.
    """
    graph = build_fk_graph(foreign_keys)
    expanded = set(selected_tables)

    for i, t1 in enumerate(selected_tables):
        for t2 in selected_tables[i + 1:]:
            path = shortest_path(graph, t1, t2)
            expanded.update(path)  # empty list if unreachable -- fine, just adds nothing

    return list(expanded)


def render_schema_context(kb: dict, selected_tables: list) -> str:
    """Renders the final compact schema text for the prompt: one CREATE
    TABLE-style block per selected table, full column list (not just
    retrieved columns -- the model needs the whole table shape to use it
    correctly), with descriptions inline, plus a relationships section.
    """
    lines = []

    cols_by_table = {}
    for col in kb["columns"]:
        cols_by_table.setdefault(col["table"], []).append(col)

    def quote_ident(name: str) -> str:
        """Backtick-quote any identifier with a space or special character,
        matching the convention BIRD's gold SQL actually uses (e.g.
        `Free Meal Count (K-12)`). A plain identifier like CDSCode is left
        unquoted since that matches gold SQL style too -- consistency with
        gold SQL's own quoting habits matters more than always/never quoting."""
        if name.isidentifier():
            return name
        return f"`{name}`"

    for t in selected_tables:
        if t not in cols_by_table:
            continue  # defensive: shouldn't happen, but don't crash the prompt build
        lines.append(f"CREATE TABLE {quote_ident(t)} (")
        n_cols = len(cols_by_table[t])
        for i, col in enumerate(cols_by_table[t]):
            is_last = (i == n_cols - 1)
            piece = f"  {quote_ident(col['column'])} {col['type']}"
            if col["is_primary_key"]:
                piece += " PRIMARY KEY"
            sep = "" if is_last else ","  # comma goes BEFORE the comment, not after
            if col["description"]:
                piece += f"{sep}  -- {col['description']}"
            else:
                piece += sep
            lines.append(piece)
        lines.append(");\n")

    relevant_fks = [
        fk for fk in kb["foreign_keys"]
        if fk["from_table"] in selected_tables and fk["to_table"] in selected_tables
    ]
    if relevant_fks:
        lines.append("-- Relationships:")
        for fk in relevant_fks:
            lines.append(f"-- {fk['from_table']}.{fk['from_col']} = {fk['to_table']}.{fk['to_col']}")

    return "\n".join(lines)


def build_retrieved_context(kb: dict, retrieved_columns: list, top_m_tables: int = 6) -> str:
    """Full pipeline: retrieved columns -> relevant tables -> FK-expanded ->
    rendered schema text. This is the single function the generation step calls."""
    selected = aggregate_columns_to_tables(retrieved_columns, top_m_tables)
    expanded = expand_with_fk_bridges(selected, kb["foreign_keys"])
    return render_schema_context(kb, expanded)

Overwriting context_builder.py


In [42]:
import importlib
import context_builder
importlib.reload(context_builder)
from context_builder import build_retrieved_context

schema_context = build_retrieved_context(kb, results, top_m_tables=6)
print(schema_context)

CREATE TABLE frpm (
  CDSCode TEXT PRIMARY KEY,  -- CDSCode
  `Academic Year` TEXT,  -- Academic Year
  `County Code` TEXT,  -- County Code
  `District Code` INTEGER,  -- District Code
  `School Code` TEXT,  -- School Code
  `County Name` TEXT,  -- County Code 
  `District Name` TEXT,  -- District Name 
  `School Name` TEXT,  -- School Name 
  `District Type` TEXT,  -- District Type
  `School Type` TEXT,  -- School Type 
  `Educational Option Type` TEXT,  -- Educational Option Type
  `NSLP Provision Status` TEXT,  -- NSLP Provision Status
  `Charter School (Y/N)` INTEGER,  -- Charter School (Y/N)
  `Charter School Number` TEXT,  -- Charter School Number
  `Charter Funding Type` TEXT,  -- Charter Funding Type
  IRC INTEGER,
  `Low Grade` TEXT,  -- Low Grade
  `High Grade` TEXT,  -- High Grade
  `Enrollment (K-12)` REAL,  -- Enrollment (K-12)
  `Free Meal Count (K-12)` REAL,  -- Free Meal Count (K-12)
  `Percent (%) Eligible Free (K-12)` REAL,
  `FRPM Count (K-12)` REAL,  -- Free or Redu

In [43]:
retrievers_by_db = {}
kbs_by_db = {}

for db_id, kb in all_kbs.items():
    retrievers_by_db[db_id] = SchemaRetriever(all_docs[db_id]["column_docs"])
    kbs_by_db[db_id] = kb

print(f"Built {len(retrievers_by_db)} retrievers")

Built 11 retrievers


In [44]:
from schema_kb import find_value_matches
from context_builder import build_retrieved_context

PROMPT_TEMPLATE_C = """You are a SQLite expert. Given the database schema and a question, write a single SQL query that answers it.

### Schema:
{schema}

### Question:
{question}

### Evidence:
{evidence}

Return ONLY the SQL query, no explanation, no markdown formatting.

### SQL:
"""

def generate_sql_system_c(example, max_new_tokens=256, top_m_tables=6):
    db_id = example["db_id"]
    retriever = retrievers_by_db[db_id]
    kb = kbs_by_db[db_id]

    q_combined = example["question"] + " " + example.get("evidence", "")
    db_path = f"{DB_ROOT}/{db_id}/{db_id}.sqlite"
    vmatches = find_value_matches(db_path, example["question"])

    retrieved = retriever.retrieve_columns(q_combined, top_k=15, value_matches=vmatches)
    schema_context = build_retrieved_context(kb, retrieved, top_m_tables=top_m_tables)

    prompt = PROMPT_TEMPLATE_C.format(
        schema=schema_context, question=example["question"], evidence=example.get("evidence", "")
    )
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred_sql = text.strip().strip("```sql").strip("```").strip()

    return pred_sql, len(inputs["input_ids"][0])  # also return prompt token count for later latency/cost analysis

In [46]:
for ex in sample[:5]:
    pred, n_tokens = generate_sql_system_c(ex)
    print(f"Q: {ex['question'][:80]}...")
    print(f"Gold: {ex['SQL'][:100]}")
    print(f"Pred (System C): {pred[:100]}")
    print(f"Prompt tokens: {n_tokens}")
    print("---")

Q: In terms of number of points acquired, how many victories did the driver who ran...
Gold: SELECT SUM(CASE WHEN points = 91 THEN wins ELSE 0 END) FROM driverStandings
Pred (System C): SELECT SUM(wins) AS total_wins
FROM driverStandings
WHERE position = 91;
Prompt tokens: 851
---
Q: How many clients who were born in 1920 stay in east Bohemia?...
Gold: SELECT COUNT(T1.client_id) FROM client AS T1 INNER JOIN district AS T2 ON T1.district_id = T2.distri
Pred (System C): SELECT COUNT(T1.client_id) FROM client AS T1 INNER JOIN district AS T2 ON T1.district_id = T2.distri
Prompt tokens: 763
---
Q: What is the type of education offered in the school who scored the highest avera...
Gold: SELECT T2.EdOpsName FROM satscores AS T1 INNER JOIN schools AS T2 ON T1.cds = T2.CDSCode ORDER BY T1
Pred (System C): SELECT DISTINCT T2.EdOpsName FROM satscores AS T1 INNER JOIN schools AS T2 ON T1.cds = T2.CDSCode OR
Prompt tokens: 1186
---
Q: Provide list of patients and their diagnosis with triglyceride (

In [47]:
from tqdm.notebook import tqdm
import json

predictions_c = []
errors_c = []

for ex in tqdm(sample, desc="System C generation"):
    try:
        pred_sql, n_tokens = generate_sql_system_c(ex)
        predictions_c.append({
            "db_id": ex["db_id"],
            "question_id": ex["question_id"],
            "gold_sql": ex["SQL"],
            "pred_sql": pred_sql,
            "difficulty": ex["difficulty"],
            "prompt_tokens": n_tokens,
        })
    except Exception as e:
        errors_c.append({"question_id": ex["question_id"], "error": str(e)})

print(f"Generated: {len(predictions_c)}  Errors: {len(errors_c)}")
if errors_c:
    print("First few errors:", errors_c[:3])

with open("/kaggle/working/system_c_predictions.json", "w") as f:
    json.dump(predictions_c, f, indent=2)

System C generation:   0%|          | 0/200 [00:00<?, ?it/s]

Generated: 200  Errors: 0


In [49]:
from harness import score_dataset
import json

with open("/kaggle/working/system_a_predictions.json") as f:
    predictions_a = json.load(f)

results_a = score_dataset(predictions_a, db_root=DB_ROOT)
print(f"System A reloaded: EX = {results_a['execution_accuracy']:.3f}  Valid = {results_a['valid_sql_rate']:.3f}")

System A reloaded: EX = 0.396  Valid = 0.741


In [50]:
from collections import defaultdict

print("=== System A (full schema) vs System C (schema-aware retrieval) ===")
print(f"A: Execution Accuracy = {results_a['execution_accuracy']:.3f}  Valid SQL Rate = {results_a['valid_sql_rate']:.3f}")
print(f"C: Execution Accuracy = {results_c['execution_accuracy']:.3f}  Valid SQL Rate = {results_c['valid_sql_rate']:.3f}")

avg_prompt_tokens_c = sum(p["prompt_tokens"] for p in predictions_c) / len(predictions_c)
print(f"\nC average prompt tokens: {avg_prompt_tokens_c:.0f}")

by_diff_c = defaultdict(lambda: {"correct": 0, "total": 0})
for r in results_c["per_example"]:
    d = r["difficulty"]
    by_diff_c[d]["total"] += 1
    by_diff_c[d]["correct"] += r["correct"]

print("\n--- System C by difficulty ---")
for d, stats in by_diff_c.items():
    print(f"{d}: {stats['correct']}/{stats['total']} = {stats['correct']/stats['total']:.3f}")

=== System A (full schema) vs System C (schema-aware retrieval) ===
A: Execution Accuracy = 0.396  Valid SQL Rate = 0.741
C: Execution Accuracy = 0.396  Valid SQL Rate = 0.736

C average prompt tokens: 1106

--- System C by difficulty ---
simple: 50/121 = 0.413
moderate: 23/60 = 0.383
challenging: 5/19 = 0.263


In [51]:
# build question_id -> correct lookup for both systems
a_correct = {r["question_id"]: r["correct"] for r in results_a["per_example"]}
c_correct = {r["question_id"]: r["correct"] for r in results_c["per_example"]}

common_ids = set(a_correct.keys()) & set(c_correct.keys())

both_correct = sum(1 for qid in common_ids if a_correct[qid] and c_correct[qid])
both_wrong = sum(1 for qid in common_ids if not a_correct[qid] and not c_correct[qid])
a_only = sum(1 for qid in common_ids if a_correct[qid] and not c_correct[qid])  # A right, C wrong
c_only = sum(1 for qid in common_ids if not a_correct[qid] and c_correct[qid])  # C right, A wrong

print(f"Both correct:     {both_correct}")
print(f"Both wrong:       {both_wrong}")
print(f"A right, C wrong: {a_only}   <- retrieval HURT these")
print(f"C right, A wrong: {c_only}   <- retrieval HELPED these")

# McNemar's test -- is the swap pattern statistically meaningful, or just noise?
from scipy.stats import binomtest
if a_only + c_only > 0:
    result = binomtest(min(a_only, c_only), a_only + c_only, 0.5)
    print(f"\nMcNemar-style test p-value: {result.pvalue:.4f}")
    print("(p < 0.05 would suggest the swaps aren't just random noise)")
else:
    print("\nNo disagreements between A and C -- identical predictions on every question")

Both correct:     61
Both wrong:       105
A right, C wrong: 17   <- retrieval HURT these
C right, A wrong: 17   <- retrieval HELPED these

McNemar-style test p-value: 1.0000
(p < 0.05 would suggest the swaps aren't just random noise)


In [52]:
import json

comparison_summary = {
    "system_a": {"execution_accuracy": results_a["execution_accuracy"], "valid_sql_rate": results_a["valid_sql_rate"]},
    "system_c": {"execution_accuracy": results_c["execution_accuracy"], "valid_sql_rate": results_c["valid_sql_rate"],
                 "avg_prompt_tokens": avg_prompt_tokens_c},
    "paired_comparison": {
        "both_correct": both_correct, "both_wrong": both_wrong,
        "a_only_correct": a_only, "c_only_correct": c_only,
        "mcnemar_pvalue": result.pvalue if (a_only + c_only > 0) else None,
    }
}

with open("/kaggle/working/a_vs_c_comparison.json", "w") as f:
    json.dump(comparison_summary, f, indent=2)

print(json.dumps(comparison_summary, indent=2))

{
  "system_a": {
    "execution_accuracy": 0.39593908629441626,
    "valid_sql_rate": 0.7411167512690355
  },
  "system_c": {
    "execution_accuracy": 0.39593908629441626,
    "valid_sql_rate": 0.7360406091370558,
    "avg_prompt_tokens": 1105.515
  },
  "paired_comparison": {
    "both_correct": 61,
    "both_wrong": 105,
    "a_only_correct": 17,
    "c_only_correct": 17,
    "mcnemar_pvalue": 1.0
  }
}
